# Unsloth Training for Hierarchical Reasoning & Metacontroller from 'Emergent Temporaral Reasoning' Paper

### Note: Note sure I need this cell(1) now as it might mess with the new Metacontroller setup from 'ETR' Paper

# --- FIX (Not needed now i think): Disable torch.compile to avoid Dynamo Errors ---
# Probs not needed now

# Cell 1
import torch

# Monkeypatch torch.compile to be a no-op decorator/function
def no_op_compile(model=None, *args, **kwargs):
    # 1. Called as torch.compile(model, ...)
    if model is not None and callable(model):
        return model
    
    # 2. Called as @torch.compile(...) -> returns decorator
    def decorator(func):
        return func
    return decorator

torch.compile = no_op_compile
print("✅ Disabled torch.compile for stability (Robust Fix).")

In [1]:
# Cell 1: Environment Setup.


import os
os.environ["fix_mistral_regex"] = "True"
# os.environ["OMP_NUM_THREADS"] = "1"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # Extra 30% context lengths

# Install dependencies (run this if not already installed)
# !pip install unsloth vllm
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

In [2]:
# Cell 2: Set remote HF_TOKEN from local .env

import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

# ssh -i ~/.ssh/id_ed25519 dataimaginations-heirarchical-reasoning@ssh.hf.space "echo 'export HF_TOKEN={hf_token}' >> ~/.bashrc"
print("✅ Token set! Restart remote shell to activate.")

✅ Token set! Restart remote shell to activate.


In [3]:
# Cell 3: HuggingFace Login


import os
from huggingface_hub import login
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("✅ Logged in with HF_TOKEN")
else:
    login()
    print("✅ Logged in interactively")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Logged in with HF_TOKEN


In [4]:
# Cell 4: Load Model

from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 2048
lora_rank = 128      
lora_alpha = 128     # <--- Generally keep Alpha = Rank for Unsloth

print(f"⏳ Loading model with Rank {lora_rank}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3.5-mini-instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
)

print("🔗 Attaching LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,        # <--- FIX: Use the variable, don't hardcode!
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_alpha, # Set this to match the rank
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"✅ Model loaded with Rank {lora_rank}!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ Loading model with Rank 128...
==((====))==  Unsloth 2025.12.9: Fast Llama patching. Transformers: 4.57.3. vLLM: 0.13.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.484 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
🔗 Attaching LoRA adapters...


Unsloth 2025.12.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Model loaded with Rank 128!


Tuning Tips:
- If you're getting OOM: Lower NEMOTRON_SAMPLE_SIZE to 1000-2000
- If generations are too long: Lower MAX_ANSWER_TOKENS to 400
- If you want more data: Increase NEMOTRON_SAMPLE_SIZE to 5000+
The filtering keeps ~60-70% of examples typically, so 3000 samples → ~2000 usable examples mixed with your 729 HICRA examples.



# New Block for MetaController

Uses LoRa instead of 4096 x 4096 Block

### Phase 1: The Metacontroller ("The Manager")

The paper describes a specific architecture (Appendix D.2 1) that generates "controllers" ($U_t$) to modify the residual stream.

For REINFORCE-style RL training, you need to store the log probabilities during generation and use those for the policy gradient, rather than trying to backprop through the generation itself.

In [8]:
# Cell 5: Metacontroller
# NEWEST VERSION

import torch
import torch.nn as nn
import torch.nn.functional as F

# For llama it would be: `def __init__(self, embed_dim=4096, latent_dim=16, hidden_dim=64):`

class Metacontroller(nn.Module):
    def __init__(self, embed_dim=3072, latent_dim=16, hidden_dim=64): # <--- Default changed to 3072
        """
        The 'Manager' that lives inside the model.
        Args:
            embed_dim: Dimension of model's residual stream (e.g., 4096 for 8B)
            latent_dim: Size of the 'thought vector' (z)
        """
        super().__init__()
        self.latent_dim = latent_dim
        
        # 1. State Tracker (History) - Equation 12 [cite: 637]
        # Keeps track of what the model has been doing
        self.history_rnn = nn.GRUCell(embed_dim, hidden_dim)
        
        # 2. Policy Head (The Actor)
        # Decides on the next 'abstract action' (z) given history
        self.policy_mean = nn.Linear(hidden_dim, latent_dim)
        self.policy_logstd = nn.Linear(hidden_dim, latent_dim)
        
        # 3. Switching Unit (The Clock) - Equation 16 [cite: 644]
        # Decides: "Keep doing current thought" (beta=0) or "Switch to new thought" (beta=1)
        self.switch_head = nn.Sequential(
            nn.Linear(hidden_dim + latent_dim, 1),
            nn.Sigmoid()
        )
        
        # 4. Controller Decoder (Hypernetwork) - Equation 18 [cite: 649]
        # Converts the thought 'z' into actual interference vectors 'U'
        # We use Low-Rank (A and B) to save memory.
        self.rank = 8
        self.hyper_A = nn.Linear(latent_dim, embed_dim * self.rank)
        self.hyper_B = nn.Linear(latent_dim, embed_dim * self.rank)

    def forward(self, residual_input, prev_hidden, prev_z):
        """
        Runs one step of the Manager.
        """
        # Update history with current brain state of Model
        hidden = self.history_rnn(residual_input, prev_hidden)
        
        # 1. Decide if we need a new plan (Switching)
        # Check based on history and PREVIOUS plan
        switch_prob = self.switch_head(torch.cat([hidden, prev_z], dim=-1))
        
        # 2. Generate a proposal for a new plan
        mu = self.policy_mean(hidden)
        std = torch.exp(self.policy_logstd(hidden))
        dist = torch.distributions.Normal(mu, std)
        proposal_z = dist.rsample() # Sampling with reparameterization
        
        # This line is to compute log probability
        log_prob = dist.log_prob(proposal_z).sum(dim=-1)  # Sum over latent dims
        
        # 3. Temporal Integration - Equation 2 [cite: 134]
        # If switch_prob is high, take new z. If low, keep ORIGINAL z.
        # For RL, we often binarize this (Algorithm 1)
        new_z = switch_prob * proposal_z + (1 - switch_prob) * prev_z
        
        # 4. Create the intervention (The "Steering")
        # Generate LoRA matrices A and B from z
        batch_size = residual_input.shape[0]
        matrix_A = self.hyper_A(new_z).view(batch_size, -1, self.rank)
        matrix_B = self.hyper_B(new_z).view(batch_size, self.rank, -1)
        
        # The control vector U*e = B @ A @ e
        # This is the "nudge" we apply to model's brain
        return new_z, hidden, switch_prob, matrix_A, matrix_B, log_prob

### Phase 2: The Hook (Connecting Brains)

To make this work with Hugging Face, we use PyTorch Hooks. This intercepts the data flowing through layer 12 (or whichever layer you choose, paper suggests mid-depth ) and lets the Metacontroller modify it.

`model.generate()` runs inside `torch.inference_mode()`, which completely blocks gradient tracking, even for the metacontroller.

The solution is to store the inputs during generation (detached), then re-run the metacontroller afterwards in a gradient-enabled context.

In [9]:
# Cell 6: InternalRLWrapper
# New
class InternalRLWrapper:
    def __init__(self, base_model, metacontroller, target_layer=16):
        self.model = base_model
        self.meta = metacontroller
        self.target_layer = target_layer
        self.hook_handle = None
        
        # Storage for runtime states
        self.meta_hidden = None
        self.current_z = None
        
        # NEW: Store inputs for later gradient computation
        self.stored_activations = []
        self.stored_hidden_states = []
        self.stored_prev_z = []
        
    def _hook_function(self, module, input, output):
        current_activation = output[0][:, -1, :]
        
        if self.meta_hidden is None:
            batch_size = current_activation.shape[0]
            self.meta_hidden = torch.zeros(batch_size, 64, device=output[0].device)
            self.current_z = torch.zeros(batch_size, 16, device=output[0].device)

        # Store DETACHED copies of inputs for later replay
        self.stored_activations.append(current_activation.detach().clone())
        self.stored_hidden_states.append(self.meta_hidden.detach().clone())
        self.stored_prev_z.append(self.current_z.detach().clone())

        # Run Metacontroller (inference only here)
        with torch.no_grad():
            z, h, beta, A, B, _ = self.meta(current_activation, self.meta_hidden, self.current_z)
        
        self.meta_hidden = h.detach()
        self.current_z = z.detach()
        
        # Apply Intervention
        act_unsqueezed = current_activation.unsqueeze(1)
        delta = torch.bmm(torch.bmm(act_unsqueezed, A), B).squeeze(1)
        output[0][:, -1, :] = output[0][:, -1, :] + delta
        
        return output
    
    def compute_loss(self, reward):
        """Replay through metacontroller WITH gradients to compute policy loss."""
        if not self.stored_activations:
            return None
            
        total_log_prob = 0.0
        hidden = self.stored_hidden_states[0]
        prev_z = self.stored_prev_z[0]
        
        for i, activation in enumerate(self.stored_activations):
            if i > 0:
                hidden = self.stored_hidden_states[i]
                prev_z = self.stored_prev_z[i]
            
            # Re-run metacontroller WITH gradients
            _, new_hidden, _, _, _, log_prob = self.meta(activation, hidden, prev_z)
            total_log_prob = total_log_prob + log_prob
        
        # REINFORCE loss
        loss = -reward * total_log_prob.mean()
        return loss
    
    def reset_episode(self):
        self.meta_hidden = None
        self.current_z = None
        self.stored_activations = []
        self.stored_hidden_states = []
        self.stored_prev_z = []

    def register(self):
        # Attach to the specific layer - handle Unsloth/PEFT nested model structure
        # Navigate through potential wrappers (PEFT, Unsloth, etc.)
        model = self.model
        
        if hasattr(model, 'base_model'):
            model = model.base_model
        if hasattr(model, 'model'):
            model = model.model
        if hasattr(model, 'model'):
            model = model.model
        
        # Now access layers
        if hasattr(model, 'layers'):
            layer = model.layers[self.target_layer]
        else:
            raise AttributeError(f"Could not find 'layers' attribute. Model structure: {type(model)}")
        
        self.hook_handle = layer.register_forward_hook(self._hook_function)
        print(f"✅ Hook registered on layer {self.target_layer}")
        
    def remove(self):
        if self.hook_handle:
            self.hook_handle.remove()
            
    def reset_episode(self):
        self.meta_hidden = None
        self.current_z = None
        self.accumulated_log_probs = []

### Phase 3: The Training Loop (Algorithms 1 & 3)

This corresponds to Algorithm 3  in the paper. It treats the Phi model + Metacontroller as an environment.

HICRA Integration: 

This is where we reward the Metacontroller if the model outputs "Strategic Grams".


In [16]:
# Cell 7: Initialize Internal RL (The "Brain Surgery")
# NEWEST VERSION
import torch.optim as optim

# 1. THE GOLDEN RULE: Freeze the Environment (Phi-3.5)
# We are NOT training Phi right now. We are training the controller.
for param in model.parameters():
    param.requires_grad = False

print("❄️ Phi-3.5 Model is now FROZEN.")

# 2. Get the correct dimension automatically
# This handles the 3072 vs 4096 issue dynamically
hidden_size = model.config.hidden_size 
print(f"📏 Detected Hidden Size: {hidden_size}")

# 3. Initialize the Manager
# meta = Metacontroller(embed_dim=hidden_size).to("cuda")

# use this for evals `meta = Metacontroller(embed_dim=hidden_size).to("cuda").to(torch.bfloat16)`
meta = Metacontroller(embed_dim=hidden_size).to("cuda").to(torch.bfloat16)

# 4. Optimizer targets ONLY the Manager
# We use a higher LR (3e-4) because this is a tiny scratchpad network
optimizer = optim.AdamW(meta.parameters(), lr=3e-4)

# 5. Connect the Hook
# We hook into layer 16 (Middle of Phi-3.5's 32 layers)
wrapper = InternalRLWrapper(model, meta, target_layer=16)

print("🪝 Metacontroller hooked into Layer 16!")

❄️ Phi-3.5 Model is now FROZEN.
📏 Detected Hidden Size: 3072
🪝 Metacontroller hooked into Layer 16!


In [5]:
# Cell 8: Load and Combine Datasets
from datasets import load_dataset, Dataset
import json

# === Configuration ===
MAX_PROMPT_TOKENS = 400    # Filter out prompts longer than this
MAX_ANSWER_TOKENS = 600    # Filter out answers longer than this  
NEMOTRON_SAMPLE_SIZE = 3000  # How many Nemotron examples to use

# System prompt for reasoning format
SYSTEM_PROMPT = """
You are a mathematical reasoning assistant. Think through problems step by step.
Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>
"""

def format_prompt(example):
    """Format dataset for GRPO training with chat template."""
    return {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT.strip()},
            {'role': 'user', 'content': example['prompt']}
        ],
        'answer': str(example['answer'])
    }

def format_nemotron(example):
    """Convert Nemotron format to our format."""
    messages = example.get('messages', [])
    
    # Extract user prompt and assistant answer
    user_content = ""
    assistant_content = ""
    
    for msg in messages:
        if msg['role'] == 'user':
            user_content = msg['content']
        elif msg['role'] == 'assistant':
            assistant_content = msg['content']
    
    # Get expected answer (fallback to assistant content if not available)
    expected = example.get('expected_answer', '')
    if not expected:
        # Try to extract from assistant's <answer> tags if present
        if '<answer>' in assistant_content and '</answer>' in assistant_content:
            expected = assistant_content.split('<answer>')[-1].split('</answer>')[0].strip()
        else:
            expected = assistant_content[-200:] if len(assistant_content) > 200 else assistant_content
    
    return {
        'prompt': user_content,
        'answer': str(expected)
    }

def estimate_tokens(text):
    """Rough token estimate (1 token ≈ 4 chars for English)."""
    return len(str(text)) // 4

def filter_by_length(example):
    """Filter out examples that are too long."""
    prompt_tokens = estimate_tokens(example['prompt'])
    answer_tokens = estimate_tokens(example['answer'])
    return prompt_tokens <= MAX_PROMPT_TOKENS and answer_tokens <= MAX_ANSWER_TOKENS

# === 1. Load Your HICRA Synthetic Data ===
print("📂 Loading HICRA dataset...")
my_dataset = load_dataset(
    "json", 
    data_files="reasoning_dataset_v2_train.json", 
    split="train"
)
print(f"   ✅ Loaded {len(my_dataset)} HICRA examples")

# === 2. Load Nemotron Math Data (Streaming) ===
print(f"🌊 Streaming {NEMOTRON_SAMPLE_SIZE} Nemotron math examples...")
try:
    nemotron_stream = load_dataset(
        "nvidia/Nemotron-Post-Training-Dataset-v1", 
        split="math", 
        streaming=True
    )
    
    # Take a sample and convert to list
    nemotron_list = []
    for i, example in enumerate(nemotron_stream):
        if i >= NEMOTRON_SAMPLE_SIZE:
            break
        formatted = format_nemotron(example)
        # Only keep if it's not too long
        if filter_by_length(formatted):
            nemotron_list.append(formatted)
        
        if (i + 1) % 500 == 0:
            print(f"   Processed {i + 1} examples, kept {len(nemotron_list)}...")
    
    nemotron_dataset = Dataset.from_list(nemotron_list)
    print(f"   ✅ Loaded {len(nemotron_dataset)} Nemotron examples (after length filter)")
    
except Exception as e:
    print(f"   ⚠️ Could not load Nemotron: {e}")
    print("   Continuing with HICRA data only...")
    nemotron_dataset = None

# === 3. Combine Datasets ===
print("🔀 Combining datasets...")

# Filter HICRA by length too
my_dataset_filtered = my_dataset.filter(filter_by_length)
print(f"   HICRA after filter: {len(my_dataset_filtered)} examples")

if nemotron_dataset and len(nemotron_dataset) > 0:
    from datasets import concatenate_datasets
    
    # Make sure both have the same columns
    combined_dataset = concatenate_datasets([my_dataset_filtered, nemotron_dataset])
    print(f"   ✅ Combined dataset: {len(combined_dataset)} examples")
else:
    combined_dataset = my_dataset_filtered
    print(f"   ✅ Using HICRA only: {len(combined_dataset)} examples")

# === 4. Format for GRPO Training ===
print("📝 Formatting for GRPO...")
dataset_train = combined_dataset.map(format_prompt)

# Shuffle to mix the datasets
dataset_train = dataset_train.shuffle(seed=42)

# === 5. Load Test Set (HICRA only) ===
dataset_test = load_dataset(
    "json", 
    data_files="reasoning_dataset_v2_test.json", 
    split="train"
).map(format_prompt)

print(f"\n✅ Final Training Set: {len(dataset_train)} examples")
print(f"✅ Test Set: {len(dataset_test)} examples")
print(f"\nSample prompt format:")
print(dataset_train[0]['prompt'])

📂 Loading HICRA dataset...
   ✅ Loaded 729 HICRA examples
🌊 Streaming 3000 Nemotron math examples...


Resolving data files:   0%|          | 0/183 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/159 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/660 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/183 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/159 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/660 [00:00<?, ?it/s]

   Processed 500 examples, kept 498...
   Processed 1000 examples, kept 998...
   Processed 1500 examples, kept 1498...
   Processed 2000 examples, kept 1998...
   Processed 2500 examples, kept 2498...
   Processed 3000 examples, kept 2997...
   ✅ Loaded 2997 Nemotron examples (after length filter)
🔀 Combining datasets...
   HICRA after filter: 729 examples
   ✅ Combined dataset: 3726 examples
📝 Formatting for GRPO...

✅ Final Training Set: 3726 examples
✅ Test Set: 36 examples

Sample prompt format:
[{'content': 'You are a mathematical reasoning assistant. Think through problems step by step.\nRespond in the following format:\n<think>\n...\n</think>\n<answer>\n...\n</answer>', 'role': 'system'}, {'content': 'Evaluate the integral \\(\\int_0^{2\\pi} \\sqrt{\\sin^2(t) \\cos^2(t)} \\, dt\\).', 'role': 'user'}]


# in case i need this later
def extract_xml_answer(text: str) -> str:
    """Extract answer from <answer> tags."""
    if "<answer>" not in text:
        return text.strip()
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """
    Check if the model's answer matches the expected answer.
    Returns 2.0 for correct, 0.0 for incorrect.
    """
    responses = [completion[0]['content'] for completion in completions]
    extracted = [extract_xml_answer(r) for r in responses]
    
    # Debug output (first item only)
    q = prompts[0][-1]['content'][:100]  # First 100 chars of question
    print(f"---\nQ: {q}...\nExpected: {answer[0]}\nExtracted: {extracted[0][:50]}...")
    
    rewards = []
    for ext, ans in zip(extracted, answer):
        # Check if answer appears in extracted text
        if str(ans).strip() in ext:
            rewards.append(2.0)
        else:
            rewards.append(0.0)
    return rewards

def reasoning_reward_func(completions, **kwargs) -> list[float]:
    """
    HICRA-inspired reward for reasoning structure.
    Gives bonus for using strategic reasoning phrases.
    """
    responses = [completion[0]['content'] for completion in completions]
    rewards = []
    
    for response in responses:
        score = 0.0
        response_lower = response.lower()
        
        # Check for strategic grams
        for gram in STRATEGIC_GRAMS:
            if gram in response_lower:
                score += 0.05
        
        # Bonus for using reasoning tags
        if "<think>" in response and "</think>" in response:
            score += 0.2
        if "<answer>" in response and "</answer>" in response:
            score += 0.1
        
        # Cap the reward
        rewards.append(min(score, 0.5))
    
    return rewards

def format_reward_func(completions, **kwargs) -> list[float]:
    """
    Reward for correct XML format AND stopping correctly.
    """
    rewards = []
    for completion in completions:
        response = completion[0]['content']
        
        # 1. Check if it has the tags
        has_tags = "<think>" in response and "</think>" in response and "<answer>" in response and "</answer>" in response
        
        # 2. Check if it rambles after the answer
        # We split by </answer> and check if there is significant text afterwards
        parts = response.split("</answer>")
        clean_stop = False
        if len(parts) > 1:
            # If the stuff after </answer> is just whitespace or EOS, it's good.
            # If it's another <think> block, it's bad.
            remainder = parts[1].strip()
            if len(remainder) < 5: # Tolerance for tiny noise
                clean_stop = True
        
        score = 0.0
        if has_tags:
            score += 0.5
        if clean_stop:
            score += 0.5 # Big bonus for stopping!
            
        rewards.append(score)
    return rewards

print("✅ Reward functions defined")

This implements the standard REINFORCE policy gradient algorithm where you don't need to backprop through the environment (the frozen Phi model), only through the policy decisions made by the metacontroller.

Key insight: We do inference during generate() (no gradients needed), store what inputs we used, then "replay" through the metacontroller afterwards with gradients enabled to compute the policy gradient loss. This is a common pattern in RL when the environment (the frozen Phi model) doesn't need gradients.

In [12]:
# Cell 9: The "Internal HICRA" Training Loop
def extract_xml_answer(text: str) -> str:
    """Extract answer from <answer> tags."""
    if "<answer>" not in text:
        return text.strip()
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def train_internal_rl(wrapper, tokenizer, dataset, num_steps=1250):
    print("🚀 Starting Internal RL Training...")
    best_reward = 0.0
    # Enable the hook (Intervention ON)
    wrapper.register()
    
    # HICRA Strategic Grams (Your curated list)
    STRATEGIC_GRAMS = [
            # Beginning a thought
            "let's analyze", "first we need", "to solve this", "let's assume",
            
            # Logic Connectors (The most important ones)
            "implies that", "consequently", "therefore", "thus", "because", 
            "since", "given that", "conversely", "alternatively",
            
            # Process Checks (Metacognition)
            "checking the", "verifying", "double check", "but wait", "identifying",
            "notice that", "recall that", "we can conclude",
            
            # Mathematical Actions
            "substituting", "calculating", "simplifying", "solving for", "derivative of"
        ]

    for step in range(num_steps):
        # 1. Get Data (Using the 'dataset' argument passed to the function)
        # FIX: We access the row directly. It returns a Dict.
        example = dataset[step % len(dataset)] 
        
        # FIX: Access the key directly. No [0] needed!
        prompt = example['prompt'] 
        
        # 2. Prepare Inputs - Apply chat template to convert message list to string
        prompt_text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
        
        # 3. Reset Manager State (New thought for new problem)
        wrapper.reset_episode()
        
        # 4. Generate with Intervention (The Puppet Dance)
        # We perform the forward pass. The HOOK inside wrapper does the magic.
        # We generate fewer tokens (256) for speed during this training phase.
        outputs = wrapper.model.generate(
            **inputs, 
            max_new_tokens=256,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.7
        )
        
        # 5. Decode the Output - ONLY the new tokens
        input_length = inputs['input_ids'].shape[1]
        new_tokens = outputs[0][input_length:]
        generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        generated_lower = generated_text.lower()
        
        # 6. CALCULATE REWARD (Full HICRA + Correctness)
        reward = 0.0
        expected_answer = str(example['answer']).strip()
        extracted_answer = extract_xml_answer(generated_text)

        # A) CORRECTNESS - The main reward!
        if expected_answer in extracted_answer:
            reward += 2.0

        # B) Reasoning Grams (capped at 0.5)
        gram_score = 0.0
        gram_count = 0
        for gram in STRATEGIC_GRAMS:
            if gram in generated_lower:
                gram_score += 0.05
                gram_count += 1
        reward += min(gram_score, 0.5)

        # C) Format + Clean Stopping
        has_all_tags = all(tag in generated_text.lower() for tag in ["<div class=\"think\">", "</div>", "<answer>", "</answer>"])
        if has_all_tags:
            reward += 0.5
            parts = generated_text.split("</answer>")
            if len(parts) > 1 and len(parts[1].strip()) < 5:
                reward += 0.5  # Clean stop bonus!
        
            
        # 7. BACKPROPAGATION (The "Internal" Update)
        # Pseudo-loss: If reward is high, encourage the 'z's we picked.

        # NEW - Policy Gradient (REINFORCE):
        if wrapper.accumulated_log_probs:
            # Stack all log probs from this episode and sum them
            episode_log_prob = torch.stack(wrapper.accumulated_log_probs).sum()
            # Calculate loss using stored trajectory (WITH gradients)
            loss = wrapper.compute_loss(reward)

            if loss is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        # Inside the loop (after calculating reward)
        if reward >= best_reward:
            best_reward = reward
            torch.save(meta.state_dict(), "best_metacontroller.pt")
            print(f"🌟 Step {step} | New Best Reward: {reward:.2f} (Saved!)")
            print(f"Generated: {generated_text[:300]}...")  # Only print output on new best!
        
        # Reset for next episode
        wrapper.reset_episode()
        # Progress update every 50 steps (without full output)
        if step % 50 == 0:
            print(f"Step {step} | Reward: {reward:.2f} | Grams: {gram_count}")

    # Cleanup
    wrapper.remove()
    print("✅ Metacontroller Training Complete!")

# Run it!
train_internal_rl(wrapper, tokenizer, dataset_train, num_steps=1250)

🚀 Starting Internal RL Training...
✅ Hook registered on layer 16
🌟 Step 0 | New Best Reward: 0.10 (Saved!)
Generated: falling thought process:

1. I recognize that the integral involves a trigonometric identity.
2. I recall that the Pythagorean identity \(sin^2(t) + cos^2(t) = 1\).
3. I rewrite the integral using the identity: \(\sqrt{sin^2(t) cos^2(t)} = \frac{1}{2} sin(2t)\).
4. I now have a simpler integral: \(\...
Step 0 | Reward: 0.10 | Grams: 2
🌟 Step 3 | New Best Reward: 0.10 (Saved!)
Generated: Intuitively, the problem requires us to find the maximum value of \( |Z^n - 1| \) for a complex number \( Z \) lying within a triangle formed by the points \( 0 \), \( 1 \), and \( e^{i\frac{2\pi}{n}} \).

Here's a step-by0r approach to solving this problem:

<think>
To solve this problem, we need t...
🌟 Step 4 | New Best Reward: 0.15 (Saved!)
Generated: {think}
To solve this problem, we need to use the properties of triangles and the relationships between their angles and sides.

First,

The "Puppet Master" Analogy
The LLM (Phi-3.5) is the Puppet.
It has the capacity to move (generate text), but it's currently floppy and uncoordinated.

The Metacontroller is the Puppeteer.
It pulls the strings (Steering Vectors $z$) to make the puppet dance.Your Question: "Am I only training the metacontroller?"
Phase 1: When you run the train_internal_rl loop (the code we just fixed), you are only training the Puppeteer.
The Puppet (LLM) MUST be frozen.
Why? If the Puppet starts changing its own strings while the Puppeteer is learning how to pull them, the Puppeteer gets confused. The "environment" becomes unstable.
Result: At the end of Phase 1, the LLM itself is no smarter. It is exactly the same file. BUT, you now have a tiny, smart network (The Metacontroller) that knows exactly how to poke the LLM to make it produce brilliant reasoning.

Your Goal: "Train the metacontroller to help the LLM train"

Phase 2: The Knowledge Transfer (Distillation). 
You don't leave the LLM frozen forever. 
You use the Metacontroller to teach the LLM.Here is the full pipeline you are building:
1. Train Metacontroller (Internal RL): 

- LLM: Frozen.

- Meta: Training.

- Outcome: A system that produces amazing reasoning, but is slow and complex (needs hooks).

2. Generate the "Golden" Dataset:

- Run your setup: Metacontroller + Frozen LLM.

- Feed it 5,000 math problems.

- Record the outputs. Because the Metacontroller is steering, these outputs will have perfect HICRA structure and deep reasoning.

- Outcome: A dataset of "Super-Reasoning" that the base LLM could never have written on its own.

3. Train the LLM (SFT):

- Now, delete the Metacontroller. Throw it away.

- Load the LLM (Unfrozen).

- Train it on that "Golden Dataset" using standard SFT.

- Outcome: The LLM internalizes the "Puppeteer's Logic" into its own weights. 

Finish the Metacontroller training (get that reward stable).

Then, write a cell to generate text using the Metacontroller and save it to a .json file.
Finally, you can start a new run to train the LLM on that JSON.You are effectively building a "Synthetic Teacher" to train your model, which is exactly how the "Emergent Temporal Abstractions" paper suggests utilizing the learned hierarchy!

### Chat Template (Save for Base models)

```
# Set Llama 3 chat template (required for GRPO with conversational data)
tokenizer.chat_template = """{% for message in messages %}{% if message['role'] == 'system' %}<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'user' %}<|start_header_id|>user<|end_header_id|}
{{ message['content'] }}<|eot_id|>{% elif message['role'] == 'assistant' %}<|start_header_id|>assistant<|end_header_id|>
{{ message['content'] }}<|eot_id|>{% endif %}{% endfor %}{% if add_generation_prompt %}<|start_header_id|>assistant<|end_header_id|>
{% endif %}"""
print("✅ Chat template set!")
```

**Optional: Use More GPU**
You could also try:

- `num_generations=6` (more diverse rollouts per step)
- Or increase `max_seq_length=1280` in cell 3 if Nemotron answers are very long

### How to calculate the Math:

Base Model Cost: A 4-bit model takes ~0.7 GB per billion parameters.Phi-3.5 (3.8B) $\approx$ 2.5 GB.Phi-4 (14B) $\approx$ 10 GB.

Context Cost (The Killer): This is determined by num_generations $\times$ max_completion_length. 16 gens $\times$ 1536 tokens is a lot of data.

The Formula: If nvidia-smi (or equiv for whatever you use) says you are only using 12GB / 24GB, double your num_generations. 

This is the safest way to improve performance without changing the model architecture.

Restart your kernel, run cells 1-9, then run your resume training cell. 🚀



In [ ]:
# Cell 13: Run Training!

print("🏋️ Resuming training from checkpoint...")

# Option A: Resume from the latest checkpoint automatically
# trainer.train(resume_from_checkpoint=True)

# Option B: Resume from a specific checkpoint (if you want to go back in time)
trainer.train(resume_from_checkpoint="./phi-3.5-hicra-meta-reasoner/checkpoint-500")

print("="*50)
print("✅ Training complete!")

## Test the Trained Model

In [13]:
# Cell 16: Test Inference with Metacontroller

from unsloth import FastLanguageModel
import torch

# Put model in inference mode
FastLanguageModel.for_inference(model)

# Load the trained Metacontroller
meta.load_state_dict(torch.load("best_metacontroller.pt"))
meta.eval()
print("✅ Loaded best metacontroller weights!")

# Make sure hook is registered
wrapper.register()

# Test question
test_question = "A loan is repaid with 20 equal annual payments. The interest portion of the 16th payment is 400 and the interest portion of the 11th payment is 600. Find the interest portion of the 1st payment."

messages = [
    {"role": "system", "content": SYSTEM_PROMPT.strip()},
    {"role": "user", "content": test_question}
]

# Tokenize
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

attention_mask = torch.ones_like(inputs)

print(f"Input shape: {inputs.shape}")  # Debug: see input size

# Generate with metacontroller intervention
print("--- Generating with Metacontroller ---")
for i in range(4):
    print(f"Starting attempt {i+1}...")  # Debug: confirm loop entry
    
    # Reset metacontroller state
    wrapper.reset_episode()
    
    try:
        with torch.no_grad():  # Inference mode
            output = model.generate(
                input_ids=inputs,
                attention_mask=attention_mask,
                max_new_tokens=1024,  # Reduced for faster testing
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        print(f"Output shape: {output.shape}")  # Debug: see what came back
        
        # Decode - output is (1, seq_len), so take [0] first
        new_tokens = output[0, inputs.shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)
        print(f"\nAttempt {i+1}:\n{response[:2000]}")
        
    except Exception as e:
        print(f"Error on attempt {i+1}: {e}")

# Cleanup
wrapper.remove()
print("\n✅ Done!")

✅ Loaded best metacontroller weights!
✅ Hook registered on layer 16
Input shape: torch.Size([1, 107])
--- Generating with Metacontroller ---
Starting attempt 1...
Output shape: torch.Size([1, 453])

Attempt 1:
ma.

<think>
1. The interest portion of a loan payment decreases over time as the principal balance is paid down.

2. The interest portion for a given payment is a function of the outstanding principal at the start of that payment.

3. As the borrower makes payments, the principal decreases, and thus the interest portion of subsequent payments decreases.

4. The interest portion of the 16th payment is 400, and the interest portion of the 11th payment is 600.

5. To find the interest portion of the 1st payment, we need to consider the decrease in the interest portion from the 11th to the 16th payment.

6. The decrease in the interest portion from the 11th to the 16th payment is 600 - 400 = 200.

7. Since this decrease happens over 5 payments (from the 11th to the 16th), we can ass

# Bench for unsloth_HICRA

In [17]:
# Cell 17: Evaluate Model with Metacontroller

import lm_eval
from lm_eval.models.huggingface import HFLM

print("⏳ Setting up model for evaluation...")

# 1. Make sure metacontroller is loaded and hook is registered
# After loading weights, convert to bfloat16 to match the model
meta.load_state_dict(torch.load("best_metacontroller.pt"))
meta = meta.to(torch.bfloat16)  
meta.eval()
wrapper.register()  # Hook is now active!
print("✅ Metacontroller hook registered!")

# 2. Wrap YOUR model (not loading from path)
# HFLM can accept a model object directly
llm = HFLM(
    pretrained=model,  # Pass your actual model with hook attached!
    tokenizer=tokenizer,
    batch_size=1,  # Keep at 1 to avoid metacontroller state issues
    trust_remote_code=True,
)

# Rest of your code stays the same...
task_list = [
    "arc_challenge",
    "hellaswag", 
    "winogrande",
    "piqa",
    "mmlu",
    "gsm8k",
    "truthfulqa_mc2",
]

print(f"🚀 Running evaluation on: {task_list}...")
results = lm_eval.simple_evaluate(
    model=llm,
    tasks=task_list,
    num_fewshot=0,
    limit=50,  # Start small to test it works!
    log_samples=True,
)

from lm_eval.utils import make_table
print(make_table(results))

# Save results
import json
with open("Phi-3_5-metacontroller_benchmark_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

# Cleanup
wrapper.remove()

[lm_eval.models.huggingface|WARNING]`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
[lm_eval.models.huggingface|WARNING]Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


⏳ Setting up model for evaluation...
✅ Hook registered on layer 16
✅ Metacontroller hook registered!
🚀 Running evaluation on: ['arc_challenge', 'hellaswag', 'winogrande', 'piqa', 'mmlu', 'gsm8k', 'truthfulqa_mc2']...


[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of gsm8k from 5 to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_abstract_algebra from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_anatomy from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_astronomy from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_college_biology from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_college_chemistry from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_college_computer_science from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_college_mathematics from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_college_physics from None to 0
[lm_eval.evaluator|WARNING]Overwriting default num_fewshot of mmlu_computer_security from None to 0
[lm_eval.evaluator|WARNING]Overwri

RuntimeError: expected mat1 and mat2 to have the same dtype, but got: c10::BFloat16 != float

# Soft VRAM clear

In [ ]:
import torch
import gc

# 1. Delete the Python variables holding the model
# (Wrap in try/except so it doesn't crash if they are already gone)
try:
    del model
    del tokenizer
    del trainer
except NameError:
    print("Variables already deleted or not defined.")

# 2. Python Garbage Collection (Clears CPU RAM)
gc.collect()

# 3. PyTorch Cache Clearing (The most important step for VRAM)
torch.cuda.empty_cache()

# Verify: Print current memory usage
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU Memory Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")